<div dir="rtl" style="text-align:right">
<h1 style="text-align:right">مشتق خودکار؛ وزن هنوز ثابت است</h1>
<p style="text-align:right">درس 21 از 92 · مشتق خودکار چه چیزی را ثبت می‌کند؟ · <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">17-autograd</code></p>
<p style="text-align:right"><a target="_self" href="http://127.0.0.1:8000/part-03/chapter-03/17-autograd.html">📖 بازگشت به همین درس</a></p>
<p style="text-align:right">ثبت <bdi dir="ltr">Graph</bdi>، محاسبهٔ <bdi dir="ltr">Gradient</bdi> و تغییر وزن را با سه شاهد جدا بررسی کنید.</p><p style="text-align:right">پیش‌نیاز: <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">12-chain</code> و کار با <bdi dir="ltr">Tensor</bdi>های اعشاری؛ <bdi dir="ltr">Autograd</bdi> در درس جاری.</p>
<p style="text-align:right">زمان یادگیری درس همراه با همین دفتر: حدود ۶۰–۱۰۵ دقیقه. زمان دفتر دوباره به زمان درس اضافه نمی‌شود؛ نصب و تمرین اختیاری جداست.</p>
<p style="text-align:right">این دفتر نیمهٔ عملی درس است. مثال‌ها آمادهٔ اجرا هستند؛ دو <bdi dir="ltr">Cell</bdi> با برچسب <bdi dir="ltr">TODO</bdi> را خودتان کامل کنید. پیام <bdi dir="ltr">INCOMPLETE</bdi> یعنی هنوز چیزی ننوشته‌اید، نه اینکه پاسخ درست است. جواب مرجع در این دفتر پنهان نشده است.</p>
<p style="text-align:right">از بالا به پایین اجرا کنید. پس از تغییر هر تابع، <bdi dir="ltr">Cell</bdi> آن و سپس <bdi dir="ltr">Cell</bdi> آزمون را دوباره اجرا کنید. برای بررسی نهایی، از منوی <code style="direction:ltr;text-align:left;unicode-bidi:isolate">Kernel → Restart Kernel and Run All Cells</code> استفاده کنید.</p>
</div>

In [ ]:
from pathlib import Path
import os
import sys

project_root = next((p for p in (Path.cwd(), *Path.cwd().parents)
                     if (p / "mini_gpt").is_dir() and (p / "book_src").is_dir()), None)
if project_root is None:
    raise RuntimeError("Extract the complete learning project; open this notebook inside it.")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
print("Python:", sys.executable)
print("Project:", project_root)

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">قبل از اجرا، پیش‌بینی کنید</h2>
<p style="text-align:right">پس از <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">backward</code> برای <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">(2w-5)²</code> در <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">w=1</code>، مقدار <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">w</code> چیست و <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">w.grad</code> چیست؟ اجرای تازهٔ دوباره بدون پاک‌کردن <bdi dir="ltr">Gradient</bdi> چه می‌کند؟</p>
</div>

<div dir="rtl" style="text-align:right"><p style="text-align:right">پیش‌بینی من: …</p></div>

In [ ]:
import torch
torch.set_num_threads(1)
print("Reference: w=1, input=2, target=5")

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">این بار شما کد بنویسید</h2>
<p style="text-align:right">تابع <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">inspect_gradient(value)</code> <bdi dir="ltr">Tensor</bdi> تک‌عنصری تازه با <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">requires_grad=True</code> بسازد، <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">Loss=(2w-5)²</code> را <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">backward</code> کند و <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">tuple</code>ِ عددی <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">(loss, gradient, unchanged_weight)</code> برگرداند. هیچ به‌روزرسانی انجام ندهید.</p>
</div>

In [ ]:
def inspect_gradient(value):
    # TODO: return three Python numbers after backward
    return None

In [ ]:
def test_exercise():
    result = inspect_gradient(1.)
    if result is None:
        return False
    assert result == (9., -12., 1.)
    assert inspect_gradient(2.) == (1., -4., 2.)
    assert inspect_gradient(2.5) == (0., 0., 2.5)
    return True

exercise_complete = test_exercise()
print('PASS' if exercise_complete else 'INCOMPLETE: implement the TODO and rerun')

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">فقط یک عامل را تغییر دهید</h2>
<p style="text-align:right">فقط پاک‌کردن <bdi dir="ltr">Gradient</bdi> بین دو اجرای تازه را روشن و خاموش کنید؛ وزن در هر دو آزمایش ثابت بماند.</p>
</div>

In [ ]:
for clear in [False, True]:
    w = torch.tensor(1., requires_grad=True)
    values = []
    for _ in range(2):
        if clear:
            w.grad = None
        ((2*w-5)**2).backward()
        values.append(w.grad.item())
    print("clear:", clear, "gradients:", values, "weight:", w.item())

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">خرابی را پیدا کنید</h2>
<p style="text-align:right"><code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">item</code> برای گزارش است؛ عدد <bdi dir="ltr">Python</bdi> مسیر مشتق ندارد. تابع <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">connected_loss</code> باید <bdi dir="ltr">Loss Tensor</bdi> را برگرداند تا فراخواننده بتواند <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">backward</code> کند.</p>
</div>

In [ ]:
w = torch.tensor(1., requires_grad=True)
wrong = ((2*w-5)**2).item()
try:
    wrong.backward()
except AttributeError as error:
    print("Expected graph loss:", error)
else:
    raise AssertionError("A Python number has no backward")

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">اصلاح را خودتان بنویسید</h2>
<p style="text-align:right">علت را توضیح دهید، سپس تابع زیر را کامل کنید. خطای عمدی بالا یک نمونهٔ آموزشی است؛ آزمون پایین باید اصلاح شما را بسنجد.</p>
</div>

In [ ]:
def connected_loss(w):
    # TODO: keep the tensor connected to w
    return None

In [ ]:
def test_repair():
    result = connected_loss(w)
    if result is None:
        return False
    assert isinstance(result, torch.Tensor) and result.requires_grad
    w.grad = None
    result.backward()
    assert w.grad.item() == -12
    fresh = torch.tensor(2., requires_grad=True)
    connected_loss(fresh).backward()
    assert fresh.grad.item() == -4
    return True

repair_complete = test_repair()
print('PASS' if repair_complete else 'INCOMPLETE: implement the TODO and rerun')

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">در <bdi dir="ltr">Mini-GPT</bdi> کجا به کار می‌آید؟</h2>
<p style="text-align:right"><code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">mini_gpt/train.py</code> مقدار <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">item</code> را برای گزارش می‌گیرد، اما <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">backward</code> را روی خود <bdi dir="ltr">Tensor Loss</bdi> اجرا می‌کند. همان جدایی ساده، در مدل بزرگ هم ضروری است.</p>
</div>

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">با زبان خودتان توضیح دهید</h2>
<p style="text-align:right">اگر <bdi dir="ltr">Gradient</bdi> درست باشد ولی وزن عوض نشود، آیا مشکل الزاماً در <bdi dir="ltr">Autograd</bdi> است؟</p>
</div>
<div dir="rtl" style="text-align:right"><p style="text-align:right">پیش‌بینی و مشاهدهٔ من: …</p><p style="text-align:right">علت خرابی و اصلاح من: …</p></div>

<div dir="rtl" style="text-align:right"><p style="text-align:right"><a target="_self" href="http://127.0.0.1:8000/part-03/chapter-03/17-autograd.html">بازگشت به درس و ادامهٔ مسیر</a> · <a target="_self" href="http://127.0.0.1:8000/answers/17-autograd.html#lab-solution">فقط پس از تلاش: راه‌حل مرجع آزمایشگاه</a></p></div>